In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# 1. ARCHITECTURE: The Generative (Top-Down) Model
class PredictiveCoder(nn.Module):
    def __init__(self, latent_dim=64):
        super(PredictiveCoder, self).__init__()
        self.latent_dim = latent_dim
        
        # The 'Internal Model' that generates expectations of digits
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 784),
            nn.Sigmoid() 
        )

    def forward(self, z):
        return self.decoder(z)

In [ ]:
# 2. INFERENCE FUNCTION: Minimizing Prediction Error
def pc_inference(model, target_img, steps=100, lr=0.05):
    """
    This is the core PC mechanism. It keeps the model weights FIXED 
    and updates the latent state 'z' to minimize the Prediction Error.
    """
    model.eval()
    # Initial 'guess' or Prior
    z = torch.randn(1, model.latent_dim, requires_grad=True)
    optimizer = optim.Adam([z], lr=lr)
    
    for _ in range(steps):
        optimizer.zero_grad()
        prediction = model(z)
        
        # Prediction Error: (Actual Input - Top-down Prediction)
        error = torch.mean((prediction - target_img)**2)
        
        error.backward()
        optimizer.step()
        
    return model(z).detach(), z.detach()

In [ ]:
# 3. MAIN EXECUTION
def run_experiment():
    # Load MNIST Handwritten Digits
    transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

    # Initialize Model and Optimizer for training the 'Prior'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pc_net = PredictiveCoder().to(device)
    optimizer = optim.Adam(pc_net.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    print("Step 1: Training the model to learn handwritten digit 'Priors'...")
    # Quick training loop (1 epoch for demo purposes)
    for epoch in range(1):
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()
            
            # Simple Autoencoder training logic to build the generative weights
            # In a true PC network, this would be local error minimization
            latent_guess = torch.randn(data.size(0), 64).to(device)
            output = pc_net(latent_guess)
            loss = criterion(output, data)
            
            loss.backward()
            optimizer.step()
            if batch_idx % 200 == 0:
                print(f"Batch {batch_idx} Loss: {loss.item():.4f}")

    print("\nStep 2: Testing PC on Noisy Handwritten Input...")
    # Get a fresh handwritten digit
    test_img, _ = train_dataset[np.random.randint(0, 1000)]
    test_img = test_img.to(device)

    # Add Gaussian Noise (simulate 'shaky' or 'noisy' input)
    noisy_img = torch.clamp(test_img + 0.5 * torch.randn_size=test_img.size()).to(device), 0, 1)

    # Perform PC Inference
    reconstructed, _ = pc_inference(pc_net, noisy_img)

    # 4. VISUALIZATION
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.imshow(test_img.cpu().view(28, 28), cmap='gray')
    plt.title("Original Handwriting")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(noisy_img.cpu().view(28, 28), cmap='gray')
    plt.title("Noisy Input\n(High Prediction Error)")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(reconstructed.cpu().view(28, 28), cmap='gray')
    plt.title("PC Reconstruction\n(Error Minimized)")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_experiment()